In [8]:
# -----------------------------
# Cell 1: Imports and paths
# -----------------------------

import os
import re
import json
import time
from pathlib import Path
from getpass import getpass

import pandas as pd
from tqdm import tqdm

try:
    from google import genai
except ImportError:
    raise ImportError("Please install google-genai first: pip install google-genai")

OUTPUT_DIR = Path("region_sensitivity_outputs")
EVAL_OUTPUT_DIR = Path("region_sensitivity_eval_outputs")
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_1800_eval.json"
DEBUG_QA_PATH = OUTPUT_DIR / "region_sensitivity_qa_pairs_context_v2_debug_15.json"

print("Current working directory:", Path.cwd())
print("Official QA path:", OFFICIAL_QA_PATH)
print("Official QA exists:", OFFICIAL_QA_PATH.exists())
print("Debug QA exists:", DEBUG_QA_PATH.exists())
print("Evaluation output directory:", EVAL_OUTPUT_DIR.resolve())


Current working directory: /Users/tanghuiru/Desktop/labeled_data_v1
Official QA path: region_sensitivity_outputs/region_sensitivity_qa_pairs_context_v2_1800_eval.json
Official QA exists: True
Debug QA exists: True
Evaluation output directory: /Users/tanghuiru/Desktop/labeled_data_v1/region_sensitivity_eval_outputs


In [9]:
# -----------------------------
# Cell 2: Load QA data
# -----------------------------
# Set USE_DEBUG = True only for a tiny test set.
# Set USE_DEBUG = False for the official 1800-example evaluation.

USE_DEBUG = False

QA_PATH = DEBUG_QA_PATH if USE_DEBUG else OFFICIAL_QA_PATH

if not QA_PATH.exists():
    raise FileNotFoundError(
        f"QA file not found: {QA_PATH}\n"
        "Make sure this notebook is in the same folder where region_sensitivity_outputs exists."
    )

with open(QA_PATH, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

eval_df = pd.DataFrame(qa_data)

print("Loaded QA:", eval_df.shape)
print(eval_df["task_type"].value_counts())
print()
print("Answer distribution by task:")
print(pd.crosstab(eval_df["task_type"], eval_df["answer"]))


Loaded QA: (1800, 9)
task_type
pairwise_comparison     600
top_sensitive_region    600
management_priority     600
Name: count, dtype: int64

Answer distribution by task:
answer                  A    B    C    D
task_type                               
management_priority   162  157  150  131
pairwise_comparison   293  307    0    0
top_sensitive_region  161  144  142  153


In [10]:
# -----------------------------
# Cell 3: Check evaluation fields
# -----------------------------

required_cols = [
    "id",
    "task_type",
    "system_prompt",
    "question",
    "answer",
    "gold_region"
]

missing_cols = [col for col in required_cols if col not in eval_df.columns]

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

print("All required columns found.")
print("Columns in eval_df:")
print(eval_df.columns.tolist())

if not USE_DEBUG:
    assert len(eval_df) == 1800, (
        f"Official evaluation should use 1800 QA rows, but eval_df has {len(eval_df)} rows. "
        "If this is not intended, check USE_DEBUG and QA_PATH."
    )
    print("Official 1800-row evaluation set confirmed.")


All required columns found.
Columns in eval_df:
['id', 'task_type', 'system_prompt', 'question', 'options', 'answer', 'gold_region', 'weather_condition', 'candidate_lgas']
Official 1800-row evaluation set confirmed.


In [11]:
# -----------------------------
# Cell 4: Extract option letter from model response
# -----------------------------

def extract_option_letter(response_text):
    """
    Extract A/B/C/D from model output.

    This avoids taking the first option letter mentioned in the reasoning.
    It prioritises final-answer patterns such as:
    - "The final answer is C"
    - "Answer: C"
    - "\\boxed{C}"
    """

    if response_text is None:
        return None

    text = str(response_text).strip()
    upper_text = text.upper()

    # Direct one-letter answer
    if upper_text in ["A", "B", "C", "D"]:
        return upper_text

    upper_text = re.sub(r"\s+", " ", upper_text)

    # LaTeX boxed answer, e.g. \boxed{C}
    boxed_match = re.search(r"\\BOXED\{([ABCD])\}", upper_text)
    if boxed_match:
        return boxed_match.group(1)

    boxed_match_2 = re.search(r"BOXED\{([ABCD])\}", upper_text)
    if boxed_match_2:
        return boxed_match_2.group(1)

    # Final answer wording
    final_patterns = [
        r"THE FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"FINAL ANSWER\s*[:\-]?\s*([ABCD])",
        r"THE ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER IS\s*[:\-]?\s*([ABCD])",
        r"ANSWER\s*[:\-]?\s*([ABCD])",
        r"OPTION\s*([ABCD])",
        r"CHOOSE\s*([ABCD])",
    ]

    for pattern in final_patterns:
        matches = re.findall(pattern, upper_text)
        if matches:
            return matches[-1]

    # Direct format like "C." or "C)"
    direct_match = re.match(r"^([ABCD])[\.)]?$", upper_text)
    if direct_match:
        return direct_match.group(1)

    # Fallback: use the last standalone option letter, not the first.
    all_matches = re.findall(r"\b([ABCD])\b", upper_text)
    if all_matches:
        return all_matches[-1]

    return None


In [12]:
# -----------------------------
# Cell 5: Set Gemini API key and create client
# -----------------------------
# Do not hard-code your API key in this notebook.
# It will ask for the key securely if the environment variable is not set.

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

print("Gemini client ready.")


Gemini client ready.


In [13]:
# -----------------------------
# Cell 6: Gemini API call functions, with safe STOP_RUN logic
# -----------------------------

def call_gemini_model(system_prompt, question, model_name="gemini-2.5-pro"):
    """
    Call Gemini model and return raw text response.
    Only system_prompt and question are sent to the model.
    Gold answer fields are not sent.
    """

    full_prompt = (
        system_prompt
        + "\n\n"
        + question
        + "\n\nImportant instruction: Return only one option letter: A, B, C, or D. "
        + "Do not explain your reasoning."
    )

    response = gemini_client.models.generate_content(
        model=model_name,
        contents=full_prompt
    )

    return response.text


def call_gemini_model_with_retry(
    system_prompt,
    question,
    model_name="gemini-2.5-pro",
    max_retries=5,
    base_sleep=60
):
    """
    Call Gemini with retry logic.

    Important behavior:
    - Stop immediately when prepaid credits are depleted.
    - Stop immediately when quota/rate limit is exceeded.
    - Retry only temporary server-side errors such as 503 / 500.

    This prevents the evaluation loop from writing hundreds of failed rows when quota is exhausted.
    """

    retry_keywords = [
        "503",
        "UNAVAILABLE",
        "500",
        "INTERNAL",
        "DEADLINE_EXCEEDED"
    ]

    for attempt in range(max_retries):
        try:
            return call_gemini_model(
                system_prompt=system_prompt,
                question=question,
                model_name=model_name
            )

        except Exception as e:
            error_text = str(e)
            error_lower = error_text.lower()

            if "prepayment credits are depleted" in error_lower:
                raise RuntimeError(
                    "STOP_RUN: Gemini prepaid credits are depleted. "
                    "Top up before continuing."
                )

            if "you exceeded your current quota" in error_lower:
                raise RuntimeError(
                    "STOP_RUN: Gemini quota/rate limit exceeded. "
                    "Stop now and retry later."
                )

            should_retry = any(
                keyword in error_text
                for keyword in retry_keywords
            )

            if should_retry:
                wait_time = base_sleep * (attempt + 1)
                print(f"Temporary API error. Waiting {wait_time} seconds before retry...")
                print("Error:", error_text[:300])
                time.sleep(wait_time)
            else:
                raise e

    raise RuntimeError("Max retries exceeded due to repeated temporary API errors.")


In [ ]:
# -----------------------------
# Cell 7: Check existing Gemini result files
# -----------------------------
# Run this before resuming. It tells you which result files already exist.

for model_name in ["gemini-2.5-pro"]:
    path = EVAL_OUTPUT_DIR / f"results_{model_name}_context_v2_1800.csv"
    print("=" * 100)
    print("Model:", model_name)
    print("Path:", path)
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_csv(path)
        valid_df = df[df["error"].isna()].copy()
        error_df = df[df["error"].notna()].copy()
        print("Total rows:", len(df))
        print("Successful rows:", len(valid_df))
        print("Error rows:", len(error_df))

        if len(valid_df) > 0:
            print("Valid-only accuracy:", valid_df["is_correct"].mean())
            print("Valid rows by task:")
            print(valid_df["task_type"].value_counts())

        if len(error_df) > 0:
            print("Error rows by task:")
            print(error_df["task_type"].value_counts())
            print("First error sample:")
            print(str(error_df["error"].dropna().iloc[0])[:500])
    print()


In [15]:
# -----------------------------
# Cell 8: Single-call test before running a full resume
# -----------------------------
# Use this before Cell 9. If this fails with STOP_RUN, do NOT run the full evaluation.

TEST_MODEL_NAME = "gemini-2.5-pro"   

test_row = eval_df.iloc[0]

try:
    raw_response = call_gemini_model_with_retry(
        system_prompt=test_row["system_prompt"],
        question=test_row["question"],
        model_name=TEST_MODEL_NAME,
        max_retries=1,
        base_sleep=10
    )

    predicted_answer = extract_option_letter(raw_response)

    print("Test success.")
    print("Raw response:", raw_response)
    print("Extracted answer:", predicted_answer)
    print("Gold answer:", test_row["answer"])
    print("Is correct:", predicted_answer == test_row["answer"])

except Exception as e:
    print("Test failed. Do NOT run the official evaluation until this succeeds.")
    print(str(e)[:1000])


Test success.
Raw response: B
Extracted answer: B
Gold answer: B
Is correct: True


In [20]:
# -----------------------------
# Cell 9: Official 1800 evaluation for one Gemini model with SAFE resume support
# -----------------------------
# Before running this cell:
# 1. Cell 2 must have USE_DEBUG = False.
# 2. Cell 3 must confirm eval_df has 1800 rows.
# 3. Cell 8 single-call test should succeed.

assert len(eval_df) == 1800, (
    f"Official evaluation should use 1800 QA rows, but eval_df has {len(eval_df)} rows. "
    "Set USE_DEBUG = False and re-run Cell 2."
)

MODEL_PROVIDER = "gemini"
MODEL_NAME = "gemini-2.5-pro"   

SAVE_PATH = EVAL_OUTPUT_DIR / f"results_{MODEL_NAME}_context_v2_1800.csv"

# Conservative delay for safer Gemini 2.5 Pro runs.
DELAY_SECONDS = 30

# If previous partial results exist, load successful rows and rerun failed rows.
if SAVE_PATH.exists():
    results_df_existing = pd.read_csv(SAVE_PATH)

    successful_existing = results_df_existing[
        results_df_existing["error"].isna()
    ].copy()

    completed_ids = set(successful_existing["id"].tolist())
    results = successful_existing.to_dict("records")

    print(f"Loaded existing successful results: {len(completed_ids)} completed rows")
    print(f"Previous error rows to rerun: {len(results_df_existing) - len(successful_existing)}")

else:
    completed_ids = set()
    results = []
    print("No existing result file found. Starting fresh.")

rows_to_run = eval_df[~eval_df["id"].isin(completed_ids)].copy()

print("Total QA:", len(eval_df))
print("Already completed successfully:", len(completed_ids))
print("Remaining:", len(rows_to_run))
print("Save path:", SAVE_PATH)
print("Delay seconds:", DELAY_SECONDS)

stopped_early = False

for _, row in tqdm(rows_to_run.iterrows(), total=len(rows_to_run)):
    system_prompt = row["system_prompt"]
    question = row["question"]

    try:
        raw_response = call_gemini_model_with_retry(
            system_prompt=system_prompt,
            question=question,
            model_name=MODEL_NAME,
            max_retries=5,
            base_sleep=60
        )

        predicted_answer = extract_option_letter(raw_response)

        result_row = {
            "id": row["id"],
            "task_type": row["task_type"],
            "weather_condition": row.get("weather_condition", None),
            "model_provider": MODEL_PROVIDER,
            "model_name": MODEL_NAME,
            "gold_answer": row["answer"],
            "gold_region": row["gold_region"],
            "raw_response": raw_response,
            "predicted_answer": predicted_answer,
            "is_correct": predicted_answer == row["answer"],
            "error": None
        }

        results.append(result_row)

        # Save after every successful row.
        pd.DataFrame(results).drop_duplicates(subset=["id"], keep="last").to_csv(
            SAVE_PATH,
            index=False
        )

        time.sleep(DELAY_SECONDS)

    except Exception as e:
        # Very important: stop the whole run when quota or prepaid credits are exhausted.
        if "STOP_RUN" in str(e):
            print(str(e))
            print("Stopping evaluation to avoid writing many failed rows.")
            stopped_early = True
            break

        # Non-quota unexpected errors are recorded for later inspection/rerun.
        result_row = {
            "id": row["id"],
            "task_type": row["task_type"],
            "weather_condition": row.get("weather_condition", None),
            "model_provider": MODEL_PROVIDER,
            "model_name": MODEL_NAME,
            "gold_answer": row["answer"],
            "gold_region": row["gold_region"],
            "raw_response": None,
            "predicted_answer": None,
            "is_correct": False,
            "error": str(e)
        }

        results.append(result_row)

        pd.DataFrame(results).drop_duplicates(subset=["id"], keep="last").to_csv(
            SAVE_PATH,
            index=False
        )

results_df = pd.DataFrame(results).drop_duplicates(subset=["id"], keep="last")
results_df.to_csv(SAVE_PATH, index=False)

valid_df = results_df[results_df["error"].isna()].copy()
error_df = results_df[results_df["error"].notna()].copy()

print("Saved results to:", SAVE_PATH)
print("Total saved rows:", len(results_df))
print("Successful rows:", len(valid_df))
print("Number of errors:", len(error_df))
print("Stopped early due to STOP_RUN:", stopped_early)
print()

if len(valid_df) > 0:
    print("Valid-only accuracy:", valid_df["is_correct"].mean())
    print()
    print("Valid-only accuracy by task:")
    print(valid_df.groupby("task_type")["is_correct"].mean())
    print()
    print("Valid rows by task:")
    print(valid_df["task_type"].value_counts())

if len(error_df) > 0:
    print()
    print("Error rows by task:")
    print(error_df["task_type"].value_counts())
    print("First error sample:")
    print(str(error_df["error"].dropna().iloc[0])[:500])


Loaded existing successful results: 1603 completed rows
Previous error rows to rerun: 0
Total QA: 1800
Already completed successfully: 1603
Remaining: 197
Save path: region_sensitivity_eval_outputs/results_gemini-2.5-pro_context_v2_1800.csv
Delay seconds: 30


  5%|██                                      | 10/197 [06:55<2:13:02, 42.69s/it]

Temporary API error. Waiting 60 seconds before retry...
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


  7%|██▋                                     | 13/197 [10:49<3:02:58, 59.67s/it]

Temporary API error. Waiting 60 seconds before retry...
Error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


100%|███████████████████████████████████████| 197/197 [2:24:59<00:00, 44.16s/it]

Saved results to: region_sensitivity_eval_outputs/results_gemini-2.5-pro_context_v2_1800.csv
Total saved rows: 1800
Successful rows: 1800
Number of errors: 0
Stopped early due to STOP_RUN: False

Valid-only accuracy: 0.6511111111111111

Valid-only accuracy by task:
task_type
management_priority     0.621667
pairwise_comparison     0.776667
top_sensitive_region    0.555000
Name: is_correct, dtype: float64

Valid rows by task:
task_type
pairwise_comparison     600
top_sensitive_region    600
management_priority     600
Name: count, dtype: int64


In [21]:
# -----------------------------
# Save compact Gemini model summary
# -----------------------------

import pandas as pd
from pathlib import Path

SUMMARY_MODEL_NAME = "gemini-2.5-pro"

RESULT_PATH = (
    EVAL_OUTPUT_DIR
    / f"results_{SUMMARY_MODEL_NAME}_context_v2_1800.csv"
)

if not RESULT_PATH.exists():
    raise FileNotFoundError(f"Result file not found: {RESULT_PATH}")

results_df = pd.read_csv(RESULT_PATH)

# Clean error column
results_df["error_clean"] = (
    results_df["error"]
    .fillna("")
    .astype(str)
    .str.strip()
)

results_df["success"] = results_df["error_clean"] == ""

# Make sure is_correct is Boolean
results_df["is_correct_bool"] = (
    results_df["is_correct"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
)

summary_rows = []

# Overall summary
valid_df = results_df[results_df["success"]].copy()

summary_rows.append({
    "model_provider": "gemini",
    "model_name": SUMMARY_MODEL_NAME,
    "task_type": "overall",
    "total_rows": len(results_df),
    "successful_rows": int(results_df["success"].sum()),
    "error_rows": int((~results_df["success"]).sum()),
    "accuracy_valid_only": (
        valid_df["is_correct_bool"].mean()
        if len(valid_df) > 0
        else None
    ),
    "accuracy_errors_as_wrong": (
        results_df["is_correct_bool"].mean()
        if len(results_df) > 0
        else None
    ),
    "output_file": str(RESULT_PATH),
})

# Task-level summaries
for task_value, sub in results_df.groupby("task_type"):
    sub_valid = sub[sub["success"]].copy()

    summary_rows.append({
        "model_provider": "gemini",
        "model_name": SUMMARY_MODEL_NAME,
        "task_type": task_value,
        "total_rows": len(sub),
        "successful_rows": int(sub["success"].sum()),
        "error_rows": int((~sub["success"]).sum()),
        "accuracy_valid_only": (
            sub_valid["is_correct_bool"].mean()
            if len(sub_valid) > 0
            else None
        ),
        "accuracy_errors_as_wrong": (
            sub["is_correct_bool"].mean()
            if len(sub) > 0
            else None
        ),
        "output_file": str(RESULT_PATH),
    })

model_summary_df = pd.DataFrame(summary_rows)

SUMMARY_PATH = (
    EVAL_OUTPUT_DIR
    / "model_accuracy_summary_gemini25pro_promptmatched.csv"
)

model_summary_df.to_csv(SUMMARY_PATH, index=False)

print("Saved summary:", SUMMARY_PATH)
print("Absolute path:", SUMMARY_PATH.resolve())
print("Exists:", SUMMARY_PATH.exists())

display(model_summary_df)

Saved summary: region_sensitivity_eval_outputs/model_accuracy_summary_gemini25pro_promptmatched.csv
Absolute path: /Users/tanghuiru/Desktop/labeled_data_v1/region_sensitivity_eval_outputs/model_accuracy_summary_gemini25pro_promptmatched.csv
Exists: True


,model_provider,model_name,task_type,total_rows,successful_rows,error_rows,accuracy_valid_only,accuracy_errors_as_wrong,output_file
0,gemini,gemini-2.5-pro,overall,1800,1800,0,0.651111,0.651111,region_sensitivity_eval_outputs/results_gemini...
1,gemini,gemini-2.5-pro,management_priority,600,600,0,0.621667,0.621667,region_sensitivity_eval_outputs/results_gemini...
2,gemini,gemini-2.5-pro,pairwise_comparison,600,600,0,0.776667,0.776667,region_sensitivity_eval_outputs/results_gemini...
3,gemini,gemini-2.5-pro,top_sensitive_region,600,600,0,0.555000,0.555000,region_sensitivity_eval_outputs/results_gemini...


In [1]:
# ============================================================
# Task 5 metric verification:
# Gemini 2.5 Pro
# ============================================================

import os
import glob
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score
)

# ------------------------------------------------------------
# 1. Find all possible Gemini 2.5 Pro CSV files
# ------------------------------------------------------------

output_folder = "region_sensitivity_eval_outputs"

candidate_files = sorted(
    glob.glob(
        os.path.join(
            output_folder,
            "*gemini*pro*.csv"
        )
    ),
    key=os.path.getmtime,
    reverse=True
)

print("All possible Gemini 2.5 Pro files:\n")

row_level_candidates = []

for i, path in enumerate(candidate_files):
    print("=" * 100)
    print(f"[{i}] {os.path.basename(path)}")
    print("Modified:", pd.Timestamp(os.path.getmtime(path), unit="s"))

    try:
        temp_df = pd.read_csv(path)

        print("Shape:", temp_df.shape)
        print("Columns:", temp_df.columns.tolist())

        if "model_name" in temp_df.columns:
            print("Model names:")
            print(
                temp_df["model_name"]
                .value_counts(dropna=False)
                .head()
            )

        required_columns = {
            "gold_answer",
            "predicted_answer",
            "task_type"
        }

        if not required_columns.issubset(temp_df.columns):
            print("STATUS: Skipped because this appears to be a summary file")
            continue

        print("STATUS: Row-level result file")

        # Count errors
        if "error" in temp_df.columns:
            error_mask = (
                temp_df["error"].notna()
                & temp_df["error"].astype(str).str.strip().ne("")
            )
            error_count = int(error_mask.sum())
        else:
            error_count = 0

        # Count valid predictions
        valid_options = ["A", "B", "C", "D"]

        gold_clean = (
            temp_df["gold_answer"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        pred_clean = (
            temp_df["predicted_answer"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        valid_mask = (
            gold_clean.isin(valid_options)
            & pred_clean.isin(valid_options)
        )

        valid_count = int(valid_mask.sum())

        if valid_count > 0:
            temp_accuracy = accuracy_score(
                gold_clean[valid_mask],
                pred_clean[valid_mask]
            )
        else:
            temp_accuracy = None

        print("Error rows:", error_count)
        print("Valid predictions:", valid_count)

        if temp_accuracy is not None:
            print(
                "Accuracy on valid predictions:",
                f"{temp_accuracy * 100:.2f}%"
            )

        row_level_candidates.append({
            "path": path,
            "rows": len(temp_df),
            "errors": error_count,
            "valid": valid_count,
            "accuracy": temp_accuracy,
            "modified": os.path.getmtime(path)
        })

    except Exception as error:
        print("Could not read file:", error)


# ------------------------------------------------------------
# 2. Select the best final Gemini Pro result file
# ------------------------------------------------------------

if len(row_level_candidates) == 0:
    raise FileNotFoundError(
        "No row-level Gemini 2.5 Pro CSV was found.\n"
        "A valid file must contain gold_answer, predicted_answer "
        "and task_type."
    )

# Prefer:
# 1. 1800 rows
# 2. 1800 valid predictions
# 3. 0 errors
# 4. most recently modified
row_level_candidates = sorted(
    row_level_candidates,
    key=lambda item: (
        item["rows"] == 1800,
        item["valid"] == 1800,
        item["errors"] == 0,
        item["valid"],
        -item["errors"],
        item["modified"]
    ),
    reverse=True
)

print("\n" + "=" * 100)
print("CANDIDATE RANKING")
print("=" * 100)

for rank, item in enumerate(row_level_candidates, start=1):
    accuracy_text = (
        f"{item['accuracy'] * 100:.2f}%"
        if item["accuracy"] is not None
        else "N/A"
    )

    print(
        f"{rank}. {os.path.basename(item['path'])}\n"
        f"   rows={item['rows']}, "
        f"valid={item['valid']}, "
        f"errors={item['errors']}, "
        f"accuracy={accuracy_text}"
    )

file_path = row_level_candidates[0]["path"]

print("\n" + "=" * 100)
print("SELECTED FINAL RESULT FILE")
print("=" * 100)
print(file_path)


# ------------------------------------------------------------
# 3. Load selected file
# ------------------------------------------------------------

df = pd.read_csv(file_path)

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

if "model_name" in df.columns:
    print("\nModel names recorded in the CSV:")
    print(df["model_name"].value_counts(dropna=False))


# ------------------------------------------------------------
# 4. Confirm this is Gemini 2.5 Pro
# ------------------------------------------------------------

if "model_name" in df.columns:
    model_name_text = " ".join(
        df["model_name"]
        .dropna()
        .astype(str)
        .str.lower()
        .unique()
    )

    if "gemini" not in model_name_text:
        raise ValueError(
            "The selected file does not appear to contain Gemini results."
        )

    if "pro" not in model_name_text:
        print(
            "\nWARNING: The model_name field does not explicitly contain "
            "'pro'. Please inspect the model name above."
        )


# ------------------------------------------------------------
# 5. Standardise answers and task types
# ------------------------------------------------------------

def clean_answer(value):
    if pd.isna(value):
        return None

    answer = str(value).strip().upper()

    if answer in ["A", "B", "C", "D"]:
        return answer

    return None


df["gold_clean"] = df["gold_answer"].apply(clean_answer)
df["pred_clean"] = df["predicted_answer"].apply(clean_answer)

df["task_clean"] = (
    df["task_type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("-", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)

valid_df = df[
    df["gold_clean"].isin(["A", "B", "C", "D"])
    & df["pred_clean"].isin(["A", "B", "C", "D"])
].copy()


# ------------------------------------------------------------
# 6. Basic checks
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("BASIC CHECK")
print("=" * 70)

print("Total rows:", len(df))
print("Valid rows:", len(valid_df))
print("Invalid or empty predictions:", len(df) - len(valid_df))

if "error" in df.columns:
    error_mask = (
        df["error"].notna()
        & df["error"].astype(str).str.strip().ne("")
    )

    print("Rows with error:", int(error_mask.sum()))

print("\nTask counts in all rows:")
print(df["task_clean"].value_counts(dropna=False))

print("\nTask counts in valid rows:")
print(valid_df["task_clean"].value_counts(dropna=False))

if len(valid_df) == 0:
    raise ValueError(
        "The selected Gemini Pro CSV contains no valid A/B/C/D predictions."
    )


# ------------------------------------------------------------
# 7. Overall Accuracy and Macro-F1
# ------------------------------------------------------------

y_true = valid_df["gold_clean"]
y_pred = valid_df["pred_clean"]

overall_accuracy = accuracy_score(
    y_true,
    y_pred
)

overall_macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=["A", "B", "C", "D"],
    average="macro",
    zero_division=0
)

print("\n" + "=" * 70)
print("OVERALL RESULTS")
print("=" * 70)

print(f"Accuracy check: {overall_accuracy * 100:.2f}%")
print(f"Overall Macro-F1: {overall_macro_f1 * 100:.2f}%")


# ------------------------------------------------------------
# 8. Accuracy and Macro-F1 by question type
# ------------------------------------------------------------

task_settings = {
    "pairwise_comparison": ["A", "B"],
    "top_sensitive_region": ["A", "B", "C", "D"],
    "management_priority": ["A", "B", "C", "D"]
}

task_rows = []

for task_name, labels in task_settings.items():

    task_df = valid_df[
        valid_df["task_clean"] == task_name
    ].copy()

    if len(task_df) == 0:
        task_accuracy = None
        task_macro_f1 = None

    else:
        task_accuracy = accuracy_score(
            task_df["gold_clean"],
            task_df["pred_clean"]
        )

        task_macro_f1 = f1_score(
            task_df["gold_clean"],
            task_df["pred_clean"],
            labels=labels,
            average="macro",
            zero_division=0
        )

    task_rows.append({
        "Task Type": task_name,
        "Valid Rows": len(task_df),
        "Accuracy (%)": (
            round(task_accuracy * 100, 2)
            if task_accuracy is not None
            else None
        ),
        "Macro-F1 (%)": (
            round(task_macro_f1 * 100, 2)
            if task_macro_f1 is not None
            else None
        )
    })

task_table = pd.DataFrame(task_rows)

print("\n" + "=" * 70)
print("RESULTS BY QUESTION TYPE")
print("=" * 70)

display(task_table)


# ------------------------------------------------------------
# 9. Option-level Recall and F1
# ------------------------------------------------------------

options = ["A", "B", "C", "D"]

recall_values = recall_score(
    y_true,
    y_pred,
    labels=options,
    average=None,
    zero_division=0
)

f1_values = f1_score(
    y_true,
    y_pred,
    labels=options,
    average=None,
    zero_division=0
)

option_table = pd.DataFrame({
    "Option": options,
    "Recall (%)": [
        round(value * 100, 2)
        for value in recall_values
    ],
    "F1 (%)": [
        round(value * 100, 2)
        for value in f1_values
    ]
})

print("\n" + "=" * 70)
print("OPTION-LEVEL RECALL AND F1")
print("=" * 70)

display(option_table)


# ------------------------------------------------------------
# 10. Prediction distribution
# ------------------------------------------------------------

prediction_counts = (
    y_pred.value_counts()
    .reindex(options, fill_value=0)
)

distribution_table = pd.DataFrame({
    "Option": options,
    "Prediction Count": [
        int(prediction_counts[option])
        for option in options
    ],
    "Prediction Percentage (%)": [
        round(
            prediction_counts[option] / len(valid_df) * 100,
            2
        )
        for option in options
    ]
})

print("\n" + "=" * 70)
print("PREDICTION DISTRIBUTION")
print("=" * 70)

display(distribution_table)


# ------------------------------------------------------------
# 11. Consistency check against stored is_correct
# ------------------------------------------------------------

if "is_correct" in df.columns:
    stored_accuracy = pd.to_numeric(
        df.loc[valid_df.index, "is_correct"],
        errors="coerce"
    ).mean()

    print("\n" + "=" * 70)
    print("CONSISTENCY CHECK")
    print("=" * 70)

    print(
        f"Accuracy recalculated from answers: "
        f"{overall_accuracy * 100:.2f}%"
    )

    print(
        f"Accuracy from stored is_correct: "
        f"{stored_accuracy * 100:.2f}%"
    )

    print(
        f"Difference: "
        f"{abs(overall_accuracy - stored_accuracy) * 100:.6f} "
        "percentage points"
    )

All possible Gemini 2.5 Pro files:

[0] model_accuracy_summary_gemini25pro_promptmatched.csv
Modified: 2026-07-10 11:54:30.159304380
Shape: (4, 9)
Columns: ['model_provider', 'model_name', 'task_type', 'total_rows', 'successful_rows', 'error_rows', 'accuracy_valid_only', 'accuracy_errors_as_wrong', 'output_file']
Model names:
model_name
gemini-2.5-pro    4
Name: count, dtype: int64
STATUS: Skipped because this appears to be a summary file
[1] results_gemini-2.5-pro_context_v2_1800.csv
Modified: 2026-07-10 11:28:28.149077415
Shape: (1800, 11)
Columns: ['id', 'task_type', 'weather_condition', 'model_provider', 'model_name', 'gold_answer', 'gold_region', 'raw_response', 'predicted_answer', 'is_correct', 'error']
Model names:
model_name
gemini-2.5-pro    1800
Name: count, dtype: int64
STATUS: Row-level result file
Error rows: 0
Valid predictions: 1800
Accuracy on valid predictions: 65.11%

CANDIDATE RANKING
1. results_gemini-2.5-pro_context_v2_1800.csv
   rows=1800, valid=1800, errors=0, a

,Task Type,Valid Rows,Accuracy (%),Macro-F1 (%)
0,pairwise_comparison,600,77.67,77.63
1,top_sensitive_region,600,55.50,55.43
2,management_priority,600,62.17,62.11



OPTION-LEVEL RECALL AND F1


,Option,Recall (%),F1 (%)
0,A,66.40,67.60
1,B,69.41,68.95
2,C,61.64,59.50
3,D,56.69,57.40



PREDICTION DISTRIBUTION


,Option,Prediction Count,Prediction Percentage (%)
0,A,594,33.00
1,B,616,34.22
2,C,313,17.39
3,D,277,15.39



CONSISTENCY CHECK
Accuracy recalculated from answers: 65.11%
Accuracy from stored is_correct: 65.11%
Difference: 0.000000 percentage points
